## Multi-protocol CNN Raw CSI Experiments

This notebook runs the fixed-capacity CNN across block, LOVO, and cross-session protocols with one independent run per configured seed.

In [1]:
from __future__ import annotations

import pandas as pd
import torch

from utils.config import (
    ANCHOR_GROUPS,
    ARCHITECTURE,
    BANDS_TO_RUN,
    DATA_DIR,
    DEFAULT_CNN_PARAMS,
    DEFAULT_OVERLAP_SIZE,
    DEFAULT_WINDOW_SIZE,
    EXPECTED_ANCHORS,
    EXPECTED_SUBCARRIERS,
    SEEDS,
)
from utils.DL.dl_pipeline import (
    create_position_label_encoder,
    get_cache_path,
    get_results_path,
    prepare_dl_data,
    print_torch_environment,
    run_dl_experiments,
    show_dl_results,
)


### Configuration

In [2]:
CALIBRATION_MODE = "rssi"    # ("none", "packet_norm", "rssi")

CSV_PROCESSING_OPTIONS = {
    "max_workers": 1,
    "cache_dir": None,
    "use_cache": True,
    "force_reprocess": False,
    "min_rssi_dbm": -95.0,
}

MAGNITUDE_PROCESSING_OPTIONS = {
    "normalization": "empty_baseline",  # none | zscore | minmax | packet_minmax | empty_baseline
    "epsilon": 1e-8,
}
NORMALIZATION_BASELINE_SCOPE = "per_session"

FEATURE_EXTRACTION_OPTIONS = {
    "window_size": DEFAULT_WINDOW_SIZE,
    "overlap_size": DEFAULT_OVERLAP_SIZE,
    "require_all_esps": False,
}

BLOCK_COUNT = 10
TEST_SIZE = 0.30
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42
FORCE_RETRAIN = False
SPLIT_MODES = ("lovo",)

CNN_PARAMS = {
    **DEFAULT_CNN_PARAMS,
    "model_label": "CNN",
    "epochs": 70,
    "patience": 15,
    "lr": 1e-3,
    "weight_decay": 1e-4,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "validation_size": VALIDATION_SIZE,
    "n_blocks": BLOCK_COUNT,
    "anchor_groups": ANCHOR_GROUPS,
    "torch_version": torch.__version__,
}

preproc_opts = dict(MAGNITUDE_PROCESSING_OPTIONS)
preproc_opts["baseline_scope"] = NORMALIZATION_BASELINE_SCOPE
feat_opts = dict(FEATURE_EXTRACTION_OPTIONS)

feature_cache_dir = get_cache_path(preproc_opts, feat_opts)
results_dir = get_results_path()
plots_dir = results_dir / "plots"
for directory in (plots_dir, results_dir / "predictions", results_dir / "tables"):
    directory.mkdir(parents=True, exist_ok=True)

print(f"Feature cache path: {feature_cache_dir}")
print(f"Results path: {results_dir}")


Feature cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0
Results path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results


### Environment

In [3]:
DEVICE = print_torch_environment(require_cuda=True)


torch.__version__: 2.11.0+cu128
torch.version.cuda: 12.8
torch.cuda.get_device_name(0): NVIDIA RTX PRO 6000 Blackwell Max-Q Workstation Edition
torch.cuda.get_device_capability(0): (12, 0)
CUDA matmul smoke test result: [[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0], [8.0, 9.0, 10.0, 11.0], [12.0, 13.0, 14.0, 15.0]] (PASS)


### Data

In [4]:
processed_magnitude_data, feature_dataframes, csv_diagnostics, magnitude_summary = prepare_dl_data(
    DATA_DIR,
    calibration_mode=CALIBRATION_MODE,
    csv_options=CSV_PROCESSING_OPTIONS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
)
display(magnitude_summary.head())


Scenarios present: 1
Locations: 53 | Users: 6 | ESPs: 19
[Z-0 inventory]
  total Z-0 files found: 171
  Z-0 count per (user, trial):
    (user=01, trial=01): 19
    (user=01, trial=02): 19
    (user=02, trial=01): 19
    (user=03, trial=01): 19
    (user=03, trial=02): 19
    (user=04, trial=01): 19
    (user=05, trial=01): 19
    (user=05, trial=02): 19
    (user=06, trial=01): 19
  breakdown by user / esp / trial:
    user=01 esp=01 trial=01: 1
    user=01 esp=01 trial=02: 1
    user=01 esp=02 trial=01: 1
    user=01 esp=02 trial=02: 1
    user=01 esp=03 trial=01: 1
    user=01 esp=03 trial=02: 1
    user=01 esp=04 trial=01: 1
    user=01 esp=04 trial=02: 1
    user=01 esp=05 trial=01: 1
    user=01 esp=05 trial=02: 1
    user=01 esp=07 trial=01: 1
    user=01 esp=07 trial=02: 1
    user=01 esp=08 trial=01: 1
    user=01 esp=08 trial=02: 1
    user=01 esp=09 trial=01: 1
    user=01 esp=09 trial=02: 1
    user=01 esp=10 trial=01: 1
    user=01 esp=10 trial=02: 1
    user=01 esp=11 tri

,scenario,location,user,esp,trial,samples,subcarriers,normalization,baseline_scope
0,1,C-1,06,16,01,2088,56,empty_baseline,per_session
1,1,C-1,06,15,01,1942,56,empty_baseline,per_session
2,1,C-1,06,09,01,1689,50,empty_baseline,per_session
3,1,C-1,06,07,01,1965,50,empty_baseline,per_session
4,1,C-1,06,04,01,1913,50,empty_baseline,per_session


In [5]:
for band in BANDS_TO_RUN:
    df = feature_dataframes[band]
    print(f"{band}: {df.shape[0]} windows, {df.shape[1]} columns")
    print(f"{band} dataframe hash: {pd.util.hash_pandas_object(df, index=True).sum()}")


2.4 GHz: 13588 windows, 2708 columns
2.4 GHz dataframe hash: 12830868654705884127
5 GHz: 14478 windows, 3368 columns
5 GHz dataframe hash: 5016803871464446121
Fusion: 13568 windows, 6068 columns
Fusion dataframe hash: 3976504561903335248


In [6]:
label_encoder = create_position_label_encoder(
    feature_dataframes,
    results_dir=results_dir,
    expected_classes=52,
)
print(label_encoder.classes_)


[CNN] label classes saved to /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/manifests/cnn_label_classes.json
['A-1' 'A-13' 'A-14' 'A-2' 'A-5' 'B-1' 'B-10' 'B-11' 'B-12' 'B-13' 'B-14'
 'B-2' 'B-5' 'B-8' 'C-1' 'C-10' 'C-11' 'C-14' 'C-2' 'C-3' 'C-4' 'C-5'
 'C-6' 'C-7' 'C-8' 'C-9' 'D-1' 'D-2' 'D-3' 'D-4' 'D-5' 'D-6' 'D-7' 'D-8'
 'E-1' 'E-10' 'E-11' 'E-12' 'E-13' 'E-2' 'E-3' 'E-4' 'E-5' 'E-6' 'E-7'
 'E-8' 'F-10' 'F-11' 'F-12' 'F-13' 'F-5' 'F-8']


### Train And Evaluate

In [7]:
cnn_runs = run_dl_experiments(
    processed_magnitude_data,
    feature_dataframes,
    bands=BANDS_TO_RUN,
    split_modes=SPLIT_MODES,
    label_encoder=label_encoder,
    device=DEVICE,
    results_dir=results_dir,
    plots_dir=plots_dir,
    params=CNN_PARAMS,
    preproc_opts=preproc_opts,
    feat_opts=feat_opts,
    expected_subcarriers=EXPECTED_SUBCARRIERS,
    expected_anchors=EXPECTED_ANCHORS,
    architecture=ARCHITECTURE,
    seeds=SEEDS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    n_blocks=BLOCK_COUNT,
    val_size=VALIDATION_SIZE,
    force_retrain=FORCE_RETRAIN,
)



=== Fusion ===
[window arrays] cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/window_arrays/fusion
[window arrays cache hit] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session/feat=win60-step0/window_arrays/fusion
[window arrays] 2.4 GHz: shape=(13568, 9, 50, 60), dtype=float16
[window arrays] 2.4 GHz anchors: ['esp_01', 'esp_02', 'esp_03', 'esp_04', 'esp_05', 'esp_07', 'esp_08', 'esp_09', 'esp_10']
[window arrays] 5 GHz: shape=(13568, 10, 56, 60), dtype=float16
[window arrays] 5 GHz anchors: ['esp_11', 'esp_12', 'esp_13', 'esp_14', 'esp_15', 'esp_16', 'esp_17', 'esp_18', 'esp_19', 'esp_20']
[room window arrays] padding rows=2, derived from first-conv kernel_size - 1 (3 - 1).
[room window arrays] cache path: /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/.cache/dataframes/preproc=norm-empty_baseline_scope-per_session

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[CNN_room] Fusion/lovo fold=01 epoch 01/70: train_loss=3.7457 train_acc=0.0927 val_loss=3.7637 val_acc=0.0908 seconds=4.5
[CNN_room] Fusion/lovo fold=01 epoch 02/70: train_loss=3.0615 train_acc=0.1796 val_loss=3.4108 val_acc=0.1621 seconds=1.3
[CNN_room] Fusion/lovo fold=01 epoch 03/70: train_loss=2.6146 train_acc=0.2536 val_loss=3.6376 val_acc=0.1466 seconds=1.2
[CNN_room] Fusion/lovo fold=01 epoch 04/70: train_loss=2.3882 train_acc=0.2877 val_loss=3.5196 val_acc=0.1528 seconds=1.0
[CNN_room] Fusion/lovo fold=01 epoch 05/70: train_loss=2.2275 train_acc=0.3338 val_loss=3.6866 val_acc=0.1652 seconds=1.0
[CNN_room] Fusion/lovo fold=01 epoch 06/70: train_loss=2.1102 train_acc=0.3679 val_loss=4.0067 val_acc=0.1412 seconds=1.0
[CNN_room] Fusion/lovo fold=01 epoch 07/70: train_loss=1.9919 train_acc=0.3942 val_loss=3.9873 val_acc=0.1404 seconds=1.0
[CNN_room] Fusion/lovo fold=01 epoch 08/70: train_loss=1.8870 train_acc=0.4237 val_loss=4.1775 val_acc=0.1350 seconds=1.0
[CNN_room] Fusion/lovo f

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s42__dc4ee3__fold-01.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn_room__fusion__lovo__ebl-session__s42__dc4ee3/training_curves__fold-01.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL LOVO] held_out_user=02 validation_user=01 (entire user, deterministic rotation)
[CNN_room] Fusion/lovo fold=02 parameters=195636
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}


...

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc__fold-05.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc/training_curves__fold-05.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[DL LOVO] held_out_user=06 validation_user=05 (entire user, deterministic rotation)
[CNN_room] Fusion/lovo fold=06 parameters=195636
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}
[DataLoader] device=cuda settings={'batch_size': 256, 'num_workers': 8, 'pin_memory': True, 'persistent_workers': True, 'prefetch_factor': 4}


/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[CNN_room] Fusion/lovo fold=06 epoch 01/70: train_loss=3.7286 train_acc=0.0780 val_loss=3.9653 val_acc=0.0397 seconds=4.2
[CNN_room] Fusion/lovo fold=06 epoch 02/70: train_loss=3.1016 train_acc=0.1585 val_loss=3.7713 val_acc=0.1285 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 03/70: train_loss=2.7539 train_acc=0.2146 val_loss=3.4653 val_acc=0.1467 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 04/70: train_loss=2.5356 train_acc=0.2578 val_loss=3.2329 val_acc=0.1676 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 05/70: train_loss=2.3698 train_acc=0.3054 val_loss=3.2062 val_acc=0.1958 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 06/70: train_loss=2.2219 train_acc=0.3328 val_loss=3.1586 val_acc=0.2127 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 07/70: train_loss=2.0986 train_acc=0.3636 val_loss=3.0397 val_acc=0.2241 seconds=1.0
[CNN_room] Fusion/lovo fold=06 epoch 08/70: train_loss=1.9976 train_acc=0.3903 val_loss=3.0376 val_acc=0.2153 seconds=1.0
[CNN_room] Fusion/lovo f

/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  band: torch.as_tensor(array[array_index], dtype=torch.float32)
/home/pedro.monteiro@co.it.pt/Desktop/thesis-project/utils/DL/dl_pipeline.py:329: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Tr

[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc__fold-06.parquet
[plots] saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/plots/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc/training_curves__fold-06.png at dpi=200. PNG is raster; set PLOT_FORMAT='pdf' for a LaTeX vector figure without other code changes.
[predictions] Saved /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/predictions/dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc.parquet
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs_folds.csv: 72 row(s)
[DL LOVO] seed=44 fold mean position_accuracy=0.1263 +/- 0.0399; pooled=0.1275
[DL GPU] run_id=dl__cnn_room__fusion__lovo__ebl-session__s44__5ccebc peak_cuda_memory_bytes=2685729792
[results upsert] /home/pedro.monteiro@co.it.pt/Desktop/thesis-project/results/runs.csv: 42 row(s)
[DL seeds] mean +/- std across se

In [8]:
cnn_summary, test_accuracy_comparison = show_dl_results(results_dir)
display(cnn_summary)

display(test_accuracy_comparison)


,run_id,timestamp,family,model,band,split,seed,normalization,baseline_scope,window_size,...,best_epoch_std,mean_seconds_per_epoch,mean_seconds_per_epoch_mean,mean_seconds_per_epoch_std,stopped_epoch,patience_triggered,peak_cuda_memory_bytes,device,sklearn_version,numpy_version
3,dl__cnn__2_4ghz__block__ebl-session__s42__df21ec,2026-07-22T18:36:31.182108+00:00,dl,cnn,2_4ghz,block,42,empty_baseline,session,60,...,NaN,0.282136,NaN,NaN,44.0,1.0,5.147146e+08,cuda,1.9.0,2.5.1
4,dl__cnn__2_4ghz__block__ebl-session__s43__210c56,2026-07-22T18:36:47.242239+00:00,dl,cnn,2_4ghz,block,43,empty_baseline,session,60,...,NaN,0.251184,NaN,NaN,50.0,0.0,5.147146e+08,cuda,1.9.0,2.5.1
5,dl__cnn__2_4ghz__block__ebl-session__s44__86f94f,2026-07-22T18:37:03.908150+00:00,dl,cnn,2_4ghz,block,44,empty_baseline,session,60,...,NaN,0.255270,NaN,NaN,50.0,0.0,5.147146e+08,cuda,1.9.0,2.5.1
6,dl__cnn__5ghz__block__ebl-session__s42__df21ec,2026-07-22T18:37:26.695344+00:00,dl,cnn,5ghz,block,42,empty_baseline,session,60,...,NaN,0.345880,NaN,NaN,40.0,1.0,5.790833e+08,cuda,1.9.0,2.5.1
7,dl__cnn__5ghz__block__ebl-session__s43__210c56,2026-07-22T18:37:44.903111+00:00,dl,cnn,5ghz,block,43,empty_baseline,session,60,...,NaN,0.327006,NaN,NaN,44.0,1.0,5.790833e+08,cuda,1.9.0,2.5.1
8,dl__cnn__5ghz__block__ebl-session__s44__86f94f,2026-07-22T18:38:04.402553+00:00,dl,cnn,5ghz,block,44,empty_baseline,session,60,...,NaN,0.312220,NaN,NaN,50.0,0.0,5.790833e+08,cuda,1.9.0,2.5.1
9,dl__cnn__fusion__block__ebl-session__s42__df21ec,2026-07-22T18:38:41.941273+00:00,dl,cnn,fusion,block,42,empty_baseline,session,60,...,NaN,0.463929,NaN,NaN,50.0,0.0,9.784187e+08,cuda,1.9.0,2.5.1
10,dl__cnn__fusion__block__ebl-session__s43__210c56,2026-07-22T18:39:07.169167+00:00,dl,cnn,fusion,block,43,empty_baseline,session,60,...,NaN,0.432133,NaN,NaN,50.0,0.0,9.784187e+08,cuda,1.9.0,2.5.1
11,dl__cnn__fusion__block__ebl-session__s44__86f94f,2026-07-22T18:39:27.186644+00:00,dl,cnn,fusion,block,44,empty_baseline,session,60,...,NaN,0.452285,NaN,NaN,36.0,1.0,9.784187e+08,cuda,1.9.0,2.5.1
12,dl__cnn__2_4ghz__block__ebl-session__s42__eb2083,2026-07-22T22:25:24.462285+00:00,dl,cnn,2_4ghz,block,42,empty_baseline,session,60,...,NaN,0.291731,NaN,NaN,50.0,0.0,5.147146e+08,cuda,1.9.0,2.5.1


,band,model,seed,position_accuracy,parameter_count
3,2_4ghz,cnn,42,0.248563,76020.0
12,2_4ghz,cnn,42,0.286101,76020.0
4,2_4ghz,cnn,43,0.316092,76020.0
13,2_4ghz,cnn,43,0.276036,76020.0
5,2_4ghz,cnn,44,0.320402,76020.0
14,2_4ghz,cnn,44,0.293457,76020.0
6,5ghz,cnn,42,0.290049,76308.0
15,5ghz,cnn,42,0.298067,76308.0
7,5ghz,cnn,43,0.307039,76308.0
16,5ghz,cnn,43,0.296661,76308.0
